## Przygotowanie danych – foundational foods

Dotyczy podstawowych produktów spożywczych, takich jak np. jabłko, wołowina itp.

In [1]:
import json
import pandas as pd

with open("raw_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Spłaszczenie listy FoundationFoods ( bo tam wszystko jest jakby zagnieżdżone w jednym "foundation foods")
df = pd.json_normalize(data['FoundationFoods'])
df.head()



,foodClass,description,foodNutrients,foodAttributes,nutrientConversionFactors,isHistoricalReference,ndbNumber,dataType,fdcId,foodPortions,publicationDate,inputFoods,foodCategory.description,scientificName
0,FinalFood,"Hummus, commercial","[{'type': 'FoodNutrient', 'id': 2219707, 'nutr...",[],"[{'type': '.CalorieConversionFactor', 'protein...",False,16158,Foundation,321358,"[{'id': 118804, 'value': 2.0, 'measureUnit': {...",4/1/2019,"[{'id': 10428, 'foodDescription': 'HUMMUS, SAB...",Legumes and Legume Products,NaN
1,FinalFood,"Tomatoes, grape, raw","[{'type': 'FoodNutrient', 'id': 2219983, 'nutr...",[],"[{'type': '.ProteinConversionFactor', 'value':...",False,100147,Foundation,321360,"[{'id': 118808, 'value': 5.0, 'measureUnit': {...",4/1/2019,"[{'id': 10480, 'foodDescription': 'TOMATOES, G...",Vegetables and Vegetable Products,Solanum lycopersicum
2,FinalFood,"Beans, snap, green, canned, regular pack, drai...","[{'type': 'FoodNutrient', 'id': 2220526, 'nutr...",[],"[{'type': '.CalorieConversionFactor', 'protein...",False,11056,Foundation,321611,"[{'id': 118859, 'value': 1.0, 'measureUnit': {...",4/1/2019,"[{'id': 10515, 'foodDescription': 'BEANS, SNAP...",Vegetables and Vegetable Products,NaN
3,FinalFood,"Frankfurter, beef, unheated","[{'type': 'FoodNutrient', 'id': 2227684, 'nutr...",[],"[{'type': '.CalorieConversionFactor', 'protein...",False,7022,Foundation,323121,"[{'id': 118987, 'value': 1.0, 'measureUnit': {...",4/1/2019,"[{'id': 10612, 'foodDescription': 'HOT DOGS, B...",Sausages and Luncheon Meats,NaN
4,FinalFood,"Nuts, almonds, dry roasted, with salt added","[{'type': 'FoodNutrient', 'id': 2228441, 'nutr...",[],"[{'type': '.ProteinConversionFactor', 'value':...",False,12563,Foundation,323294,"[{'id': 119012, 'value': 1.0, 'measureUnit': {...",4/1/2019,"[{'id': 10625, 'foodDescription': 'ALMONDS, DR...",Nut and Seed Products,NaN


### print etykietek 

In [2]:
from pprint import pprint

flat_keys = set()
nutrient_keys = set()
derivation_keys = set()
source_keys = set()

for food in data["FoundationFoods"]:
    for item in food.get("foodNutrients", []):
        flat_keys.update(item.keys())

        if isinstance(item.get("nutrient"), dict):
            nutrient_keys.update(item["nutrient"].keys())

        if isinstance(item.get("foodNutrientDerivation"), dict):
            derivation_keys.update(item["foodNutrientDerivation"].keys())
            source = item["foodNutrientDerivation"].get("foodNutrientSource")
            if isinstance(source, dict):
                source_keys.update(source.keys())

print("🔹 Główne pola w foodNutrients:")
pprint(sorted(flat_keys))

print("\n🔸 Pola w nutrient:")
pprint(sorted(nutrient_keys))

print("\n🔸 Pola w foodNutrientDerivation:")
pprint(sorted(derivation_keys))

print("\n🔸 Pola w foodNutrientDerivation.foodNutrientSource:")
pprint(sorted(source_keys))


🔹 Główne pola w foodNutrients:
['amount',
 'dataPoints',
 'foodNutrientDerivation',
 'footnote',
 'id',
 'max',
 'median',
 'min',
 'nutrient',
 'type']

🔸 Pola w nutrient:
['id', 'name', 'number', 'rank', 'unitName']

🔸 Pola w foodNutrientDerivation:
['code', 'description', 'foodNutrientSource']

🔸 Pola w foodNutrientDerivation.foodNutrientSource:
['code', 'description', 'id']


In [3]:
def extract_paths(obj, path=""):
    paths = set()

    if isinstance(obj, dict):
        for k, v in obj.items():
            new_path = f"{path}.{k}" if path else k
            paths.update(extract_paths(v, new_path))
    elif isinstance(obj, list):
        for item in obj:
            paths.update(extract_paths(item, path))
    else:
        paths.add(path)

    return paths

# Przetwarzamy dane z jednego przykładowego produktu
sample_food = data["FoundationFoods"][0]

# Zbieramy wszystkie ścieżki
all_paths = extract_paths(sample_food)

# Wypisujemy posortowaną listę
for p in sorted(all_paths):
    print(p)


dataType
description
fdcId
foodCategory.description
foodClass
foodNutrients.amount
foodNutrients.dataPoints
foodNutrients.foodNutrientDerivation.code
foodNutrients.foodNutrientDerivation.description
foodNutrients.foodNutrientDerivation.foodNutrientSource.code
foodNutrients.foodNutrientDerivation.foodNutrientSource.description
foodNutrients.foodNutrientDerivation.foodNutrientSource.id
foodNutrients.id
foodNutrients.max
foodNutrients.median
foodNutrients.min
foodNutrients.nutrient.id
foodNutrients.nutrient.name
foodNutrients.nutrient.number
foodNutrients.nutrient.rank
foodNutrients.nutrient.unitName
foodNutrients.type
foodPortions.amount
foodPortions.gramWeight
foodPortions.id
foodPortions.measureUnit.abbreviation
foodPortions.measureUnit.id
foodPortions.measureUnit.name
foodPortions.minYearAcquired
foodPortions.modifier
foodPortions.sequenceNumber
foodPortions.value
inputFoods.foodDescription
inputFoods.id
inputFoods.inputFood.dataType
inputFoods.inputFood.description
inputFoods.inputFo

## Krótki opis etykiet na podstawie dokumentacji
#### link do doku: https://fdc.nal.usda.gov/Foundation_Foods_Documentation


---

## 🔹 Główne informacje o produkcie

| Pole | Znaczenie |
|------|-----------|
| `dataType` | Typ danych w FDC (np. `Foundation`, `SR Legacy`, `Branded`). |
| `description` | Pełna nazwa / opis żywności. |
| `fdcId` | Unikalny identyfikator rekordu w FoodData Central. |
| `foodCategory.description` | Kategoria żywności (historyczny podział z SR Legacy). |
| `foodClass` | Klasa rekordu – np. `FinalFood` (gotowy produkt) albo `Ingredient`. |

---

## 🔹 Wartości odżywcze `foodNutrients[]`

| Pole | Znaczenie |
|------|-----------|
| `foodNutrients.amount` | Zawartość składnika w 100 g produktu. |
| `foodNutrients.dataPoints` | Liczba niezależnych pomiarów użytych w analizie. |
| `foodNutrients.id` | Wewnętrzny klucz rekordu składnika. |
| `foodNutrients.type` | Typ rekordu (zwykle `FoodNutrient`). |
| `foodNutrients.max` / `min` / `median` | Statystyki wartości wśród próbek. |

**Wewnątrz `nutrient`:**

| Pole | Znaczenie |
|------|-----------|
| `foodNutrients.nutrient.id` | Kod składnika odżywczego (np. 1003 = białko). |
| `foodNutrients.nutrient.name` | Nazwa składnika (np. „Vitamin C”). |
| `foodNutrients.nutrient.number` | Numer USDA – zgodny z dawnym SR. |
| `foodNutrients.nutrient.rank` | Kolejność wyświetlania w interfejsie FDC. |
| `foodNutrients.nutrient.unitName` | Jednostka miary (g, mg, µg …). |

**Wewnątrz `foodNutrientDerivation`:**

| Pole | Znaczenie |
|------|-----------|
| `foodNutrients.foodNutrientDerivation.code` | Kod źródła danych (`A` = analiza lab., `AS` = suma). |
| `foodNutrients.foodNutrientDerivation.description` | Opis sposobu uzyskania wartości (Analytical, Estimated…). |

**Wewnątrz `foodNutrientSource`:**

| Pole | Znaczenie |
|------|-----------|
| `foodNutrientDerivation.foodNutrientSource.id` | ID pierwotnego źródła pomiaru. |
| `…code` / `…description` | Kod i opis źródła (np. „Analytical or derived from analytical”). |

---

## 🔹 Porcje `foodPortions[]`

| Pole | Znaczenie |
|------|-----------|
| `foodPortions.id` | ID rekordu porcji. |
| `foodPortions.amount` | Liczba jednostek (np. 1 szt.). |
| `foodPortions.value` / `gramWeight` | Masa odpowiadająca porcji (g). |
| `foodPortions.modifier` | Opis wielkości („1 cup”, „large slice”…). |
| `foodPortions.sequenceNumber` | Kolejność porcji w rekordzie. |
| `foodPortions.minYearAcquired` | Rok pozyskania próbki dla tej porcji. |

**Jednostka miary (`measureUnit`):**

| Pole | Znaczenie |
|------|-----------|
| `measureUnit.id` | ID jednostki (np. 999 = gram). |
| `measureUnit.name` | Nazwa („cup”, „gram”…). |
| `measureUnit.abbreviation` | Skrót („g”, „c”…). |

---

## 🔹 Skład produktu `inputFoods[]`

| Pole | Znaczenie |
|------|-----------|
| `inputFoods.id` | ID pozycji w recepturze. |
| `inputFoods.foodDescription` | Opis składnika (tekstowy). |
| `inputFoods.inputFood.fdcId` | FDC ID składnika wejściowego. |
| `inputFoods.inputFood.description` | Nazwa składnika. |
| `inputFoods.inputFood.dataType` | Typ danych składnika (Foundation, Branded…). |
| `inputFoods.inputFood.foodClass` | Klasa składnika (Ingredient, FinalFood…). |
| `inputFoods.inputFood.publicationDate` | Data publikacji rekordu składnika. |
| `inputFoods.inputFood.foodCategory.*` | Kategoria składnika (kod, opis, id). |

---

## 🔹 Współczynniki kaloryczne `nutrientConversionFactors[]`

| Pole | Znaczenie |
|------|-----------|
| `type` | Rodzaj współczynnika (`.CalorieConversionFactor`, `.ProteinConversionFactor` …). |
| `proteinValue` / `fatValue` / `carbohydrateValue` | Współczynniki kcal/g dla makroskładników. |
| `value` | Uniwersalna wartość, gdy istnieje tylko jeden współczynnik. |

---

## 🔹 Pola dodatkowe

| Pole | Znaczenie |
|------|-----------|
| `isHistoricalReference` | `True`, jeśli rekord pochodzi z archiwalnego SR Legacy. |
| `ndbNumber` | Dawny numer w bazie SR Legacy (kompatybilność). |
| `publicationDate` | Data publicznej publikacji rekordu w FDC. |




## co defacto zawiera "food portions"

Każdy produkt w Foundation Foods może mieć **kilka rekordów porcji**.  
Każdy rekord zawiera dane potrzebne do przeliczenia makroskładników na realne porcje.

### Przykład porcji – banan vs jajko

| Klucz w JSON-ie       | Co oznacza w praktyce                                  | Przykład banana                   | Przykład jajka                      |
|------------------------|----------------------------------------------------------|------------------------------------|-------------------------------------|
| `measureUnit`          | Słownikowa jednostka (`gram`, `cup`, `item`…)           | `"item"` (sztuka)                  | `"item"` (sztuka)                   |
| `value`                | „1 × measureUnit” – czyli rozmiar tej jednostki         | `1.0`                              | `1.0`                               |
| `amount`               | Ile takich jednostek opisuje porcja                     | `1.0` (1 szt.)<br>lub `0.5` (połówka) | `1.0`                               |
| `modifier`             | Opis wielkości/kształtu                                 | `"medium"` / `"large"`             | `"medium"` / `"large"`              |
| `gramWeight`           | Masa jadalnej części porcji (bez skórki/skorupki)       | `120 g` (średni banan bez skórki)  | `50 g` (duże jajko bez skorupki)    |

> Wystarczy znać `gramWeight` i przeliczyć składniki według wzoru:  
> `wartość_porcji = wartość_na_100g × (gramWeight / 100)`


In [4]:
unit_names = set()
unit_abbr  = set()
unit_ids   = set()

for food in data["FoundationFoods"]:
    for p in food.get("foodPortions", []):
        mu = p.get("measureUnit", {})
        unit_names.add(mu.get("name"))
        unit_abbr.add(mu.get("abbreviation"))
        unit_ids.add(mu.get("id"))

print("Unikalne measureUnit.name:")
pprint(sorted(filter(None, unit_names)))

print("\nUnikalne measureUnit.abbreviation:")
pprint(sorted(filter(None, unit_abbr)))

print("\nUnikalne measureUnit.id:")
pprint(sorted(filter(None, unit_ids)))

Unikalne measureUnit.name:
['Banana',
 'Onion',
 'bunch',
 'container',
 'cookie',
 'cup',
 'drumstick',
 'each',
 'egg',
 'fillet',
 'fruit',
 'link',
 'milliliter',
 'olive',
 'order',
 'oz',
 'package',
 'paired cooked w',
 'piece',
 'pieces',
 'roast',
 'serving',
 'slice',
 'spear',
 'steak',
 'tablespoon',
 'teaspoon',
 'tomatoes',
 'wedge']

Unikalne measureUnit.abbreviation:
['Banana',
 'Onion',
 'bunch',
 'ckd pr g',
 'container',
 'cookie',
 'cup',
 'drumstick',
 'each',
 'egg',
 'fillet',
 'fruit',
 'link',
 'ml',
 'olive',
 'order',
 'oz',
 'package',
 'piece',
 'pieces',
 'roast',
 'serving',
 'slice',
 'spear',
 'steak',
 'tbsp',
 'tomatoes',
 'tsp',
 'wedge']

Unikalne measureUnit.id:
[1000,
 1001,
 1002,
 1004,
 1010,
 1022,
 1023,
 1026,
 1027,
 1028,
 1033,
 1038,
 1039,
 1043,
 1044,
 1046,
 1049,
 1050,
 1054,
 1060,
 1067,
 1069,
 1071,
 1077,
 1082,
 1099,
 1117,
 1119,
 1120]


In [5]:
import json
import pandas as pd

with open("raw_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)["FoundationFoods"]

#  ID-y / nazwy wzięte wprost z dokumentacji Foundation Foods
TARGETS = {
    "kcal": {
        "prefer_id": [2047, 2048, 1008],           
        "prefer_name": [
            "Metabolizable Energy (Atwater General Factor)",
            "Metabolizable Energy (Atwater Specific Factor)",
            "Energy",
        ],
    },
    "protein_g": {
        "prefer_id": [1003],
        "prefer_name": ["Protein"],
    },
    "fat_g": {
        "prefer_id": [1004],
        "prefer_name": ["Total lipid (fat)"],
    },
    "carb_g": {
        "prefer_id": [1005],
        "prefer_name": ["Carbohydrate, by difference"],
    },
}

def pick_value(food_nutrients, prefer_id, prefer_name):
    """Zwraca amount dla pierwszego pasującego ID lub nazwy."""
    for pid in prefer_id:                      
        for item in food_nutrients:
            if item["nutrient"]["id"] == pid:
                return item["amount"]
    for pname in prefer_name:                   
        for item in food_nutrients:
            if item["nutrient"]["name"] == pname:
                return item["amount"]
    return None     # brak w produkcie

rows = []
for food in data:
    nutrients = food.get("foodNutrients", [])
    rows.append(
        {
            "fdcId": food["fdcId"],
            "description": food["description"],
            "kcal_per_100g": pick_value(nutrients, **TARGETS["kcal"]),
            "protein_g_per_100g": pick_value(nutrients, **TARGETS["protein_g"]),
            "fat_g_per_100g": pick_value(nutrients, **TARGETS["fat_g"]),
            "carb_g_per_100g": pick_value(nutrients, **TARGETS["carb_g"]),
        }
    )

df = pd.DataFrame(rows)
print(df.head())       


    fdcId                                        description  kcal_per_100g  \
0  321358                                 Hummus, commercial          229.0   
1  321360                               Tomatoes, grape, raw           27.0   
2  321611  Beans, snap, green, canned, regular pack, drai...           21.0   
3  323121                        Frankfurter, beef, unheated          314.0   
4  323294        Nuts, almonds, dry roasted, with salt added          620.0   

   protein_g_per_100g  fat_g_per_100g  carb_g_per_100g  
0                7.35           17.10            14.90  
1                0.83            0.63             5.51  
2                1.04            0.39             4.11  
3               11.70           28.00             2.89  
4               20.40           57.80            16.20  


#### sanity check, czy coś nie ma >900 kcal per 100 g

In [7]:
max_row = df.loc[df["kcal_per_100g"].idxmax()]
min_row = df.loc[df["kcal_per_100g"].idxmin()]

print("Najwięcej kcal/100 g:")
print(f'  {max_row["description"]} — {max_row["kcal_per_100g"]} kcal')

print("\n Najmniej kcal/100 g:")
print(f'  {min_row["description"]} — {min_row["kcal_per_100g"]} kcal')


Najwięcej kcal/100 g:
  Oil, coconut — 833.0 kcal

 Najmniej kcal/100 g:
  Pickles, cucumber, dill or kosher dill — 12.0 kcal


#### Wygląda w porządku

#### Kolejny check, czy gdzieś carbs / protein / fats nie ma wartości ujemnej

In [8]:
invalid_rows = df[
    (df["protein_g_per_100g"] < 0)
    | (df["fat_g_per_100g"] < 0)
    | (df["carb_g_per_100g"] < 0)
]

if invalid_rows.empty:
    print("Brak rekordów z ujemnymi wartościami białka, tłuszczu lub węglowodanów.")
else:
    print(f"Wykryto {len(invalid_rows)} rekordów z błędnymi wartościami:")
    print(invalid_rows[["fdcId", "description", "protein_g_per_100g", "fat_g_per_100g", "carb_g_per_100g"]])


Wykryto 8 rekordów z błędnymi wartościami:
       fdcId                             description  protein_g_per_100g  \
316  2727566  Chicken, drumstick, meat and skin, raw                18.4   
317  2727567      Chicken, thigh, meat and skin, raw                17.1   
318  2727568       Chicken, wing, meat and skin, raw                18.4   
319  2727569     Chicken, breast, meat and skin, raw                21.4   
320  2727570                       Lamb, ground, raw                17.5   
321  2727571                      Bison, ground, raw                19.9   
325  2727575             Pork, chop, center cut, raw                22.8   
326  2727576             Pork, belly, with skin, raw                15.2   

     fat_g_per_100g  carb_g_per_100g  
316            5.94           -0.475  
317           13.40           -0.173  
318           10.60           -0.459  
319            4.78           -0.428  
320           18.60           -0.251  
321            8.88           -0.150  

#### Jak widać, niektóre produkty mają ujemną ilość węglowodanów
Ujemne wartości węglowodanów wynikają z błędów zaokrągleń w metodzie „by difference” stosowanej przez USDA — wartość ta jest obliczana jako reszta do 100 g po odjęciu innych składników, więc przy drobnych odchyleniach pomiarowych może dać wynik ujemny.

Jednakże, takie produkty mogłyby wprowadzić zamieszanie w działaniu naszej aplikacji, dlatego, z racji że te wartości ujemne są bardzo małe ( między 0 a -1 grama na 100 gramów produktów ) zdecydowałem się je zaokrąglić do zera.

In [10]:
# Korekta ujemnych wartości węglowodanów
neg_carbs = df[df["carb_g_per_100g"] < 0]

print(f"Skorygowano {len(neg_carbs)} rekordów z ujemną ilością węglowodanów (ustawiono na 0).")

df.loc[df["carb_g_per_100g"] < 0, "carb_g_per_100g"] = 0

print("ponowny check")
invalid_rows = df[
    (df["protein_g_per_100g"] < 0)
    | (df["fat_g_per_100g"] < 0)
    | (df["carb_g_per_100g"] < 0)
]

if invalid_rows.empty:
    print("Brak rekordów z ujemnymi wartościami białka, tłuszczu lub węglowodanów.")
else:
    print(f"Wykryto {len(invalid_rows)} rekordów z błędnymi wartościami:")
    print(invalid_rows[["fdcId", "description", "protein_g_per_100g", "fat_g_per_100g", "carb_g_per_100g"]])



Skorygowano 0 rekordów z ujemną ilością węglowodanów (ustawiono na 0).
ponowny check
Brak rekordów z ujemnymi wartościami białka, tłuszczu lub węglowodanów.




#### Sprawdzenie poprawności wartości energetycznej

Weryfikacja, czy liczba kcal na 100 g zgadza się ze wzorem:
kcal = fat × 9 + carbs × 4 + protein × 4

In [11]:
df["kcal_calculated"] = (
    df["fat_g_per_100g"] * 9
    + df["carb_g_per_100g"] * 4
    + df["protein_g_per_100g"] * 4
)

df["kcal_difference"] = abs(df["kcal_per_100g"] - df["kcal_calculated"])

df_incorrect = df[df["kcal_difference"] > 10]

# podejrzane rekordy to te, gdzie różnica między kcal w bazie a spodziewanymi wynosi ponad 10
df_incorrect[["description", "kcal_per_100g", "kcal_calculated", "kcal_difference"]]


,description,kcal_per_100g,kcal_calculated,kcal_difference
0,"Hummus, commercial",229.0,242.90,13.90
4,"Nuts, almonds, dry roasted, with salt added",620.0,666.60,46.60
8,"Egg, white, dried",376.0,349.53,26.47
15,"Seeds, sunflower seed kernels, dry roasted, wi...",612.0,657.30,45.30
24,"Egg, whole, dried",575.0,558.08,16.92
26,"Egg, yolk, dried",654.0,640.58,13.42
29,"Oil, coconut",833.0,895.26,62.26
34,"Olives, green, Manzanilla, stuffed with pimiento",130.0,140.54,10.54
54,"Figs, dried, uncooked",249.0,277.08,28.08
70,"Sugars, granulated",385.0,401.28,16.28


#### sanity cehck według wzoru kcal = fat × 9 + protein × 4 + carbs × 4

Wartości `kcal per 100 g` mają sens – zdecydowana większość produktów dobrze zgadza się ze wzorem:

> `kcal ≈ tłuszcz × 9 + białko × 4 + węglowodany × 4`

Rozbieżności mogą wynikać z faktu, że ten wzór to tylko przybliżenie.  
USDA stosuje dokładniejsze przeliczniki (*Atwater specific factors*) i uwzględnia np.:

- błonnik pokarmowy (który może mieć 0–2 kcal/g),
- alkohole cukrowe (niższa kaloryczność niż zwykłe węgle),
- różnice w przyswajalności składników odżywczych.

Finalnie jedank dane mają sens


### Struktura DTO produktu w Springu

Definicja `Product` w modelu w Javue wygląda tak:


```java
public class Product implements Ownable {
    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    @Column(name = "name", nullable = false)
    private String name;

    @ManyToOne(fetch = FetchType.LAZY, optional = false)
    @JoinColumn(name = "unit_id", nullable = false)
    private Unit unit;

    @Column(name = "grams_per_unit")
    private Double gramsPerUnit;

    @Column(name = "kcal_100g")
    private Double kcal100g;

    @Column(name = "protein_100g")
    private Double protein100g;

    @Column(name = "carbs_100g")
    private Double carbs100g;

    @Column(name = "fat_100g")
    private Double fat100g;

    @ManyToOne
    @JoinColumn(name = "owner_id")
    private User owner;
}
```
Definicja `Unit` wygląda następująco:

```java
public class Unit {
    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    @Column(name = "name", nullable = false)
    private String name;

    @Column(name = "symbol")
    private String symbol;
}
```

W obróbce danych nie będziemy uwzględniać owner'a.


### Jak wypełniamy `unit` i `gramsPerUnit` z danych USDA (sekcja **Weights**)

| Pole w Foundation Foods (*Weights*) | Odpowiednik w modelu/DTO |
|-------------------------------------|---------------------------|
| `gramWeight` (masa porcji „for edible material without refuse”) | `Product.gramsPerUnit` → `ProductDTO.gramsPerUnit` |
| `measureUnit.name` (np. „gram”, „cup”, „egg”, „slice”) | `Unit.name` → `ProductDTO.unit.name` |
| `measureUnit.abbreviation` (np. „g”, „cup”, „egg”) | `Unit.symbol` → `ProductDTO.unit.symbol` |
| `measureUnit.id` (identyfikator USDA) | Metadane w `ProductDTO.unit.usdaUnitId` |
| `value` (zwykle `1.0`) + opcjonalny `modifier` („medium”, „large”) | Selekcja porcji: preferowany rekord z `value == 1`; jeśli istnieje wariant `modifier == "medium"`, używany jest właśnie ten wariant |

**Fallback porcji:** jeśli w danych brak sekcji *Weights* lub odpowiedniego wpisu, przyjmowana jest porcja **1 g**  
(`unit = {"name": "gram", "symbol": "g"}`, `gramsPerUnit = 1.0`, `usdaUnitId = null`).

**Konwersja makroskładników dla porcji:**  
dla masy porcji `W` i wartości na 100 g `N100`:


**wartość na porcję = (wartość na 100 g) × (masa porcji / 100))**


**Przykładowy rekord (finalny format JSON):**
```json
{
  "usdaId": 321360,
  "name": "Tomatoes, grape, raw",
  "unit": { "usdaUnitId": 1000, "name": "cup", "symbol": "cup" },
  "gramsPerUnit": 152.0,
  "kcal100g": 27.0,
  "protein100g": 0.83,
  "carbs100g": 5.51,
  "fat100g": 0.63
}
```

Fallback 1 g (gdy brak Weights):

```json
{
  "unit": { "usdaUnitId": null, "name": "gram", "symbol": "g" },
  "gramsPerUnit": 1.0
}
```


In [28]:
import json
import pandas as pd

# 1) Wczytanie danych
with open("raw_data.json", "r", encoding="utf-8") as f:
    foods = json.load(f)["FoundationFoods"]

# 2) Definicja makroskładników (priorytety ID -> nazwa)
TARGETS = {
    "kcal": {
        "prefer_id": [2047, 2048, 1008],
        "prefer_name": [
            "Metabolizable Energy (Atwater General Factor)",
            "Metabolizable Energy (Atwater Specific Factor)",
            "Energy",
        ],
    },
    "protein_g": {"prefer_id": [1003], "prefer_name": ["Protein"]},
    "fat_g": {"prefer_id": [1004], "prefer_name": ["Total lipid (fat)"]},
    "carb_g": {"prefer_id": [1005], "prefer_name": ["Carbohydrate, by difference"]},
}

def pick_value(nutrients, prefer_id, prefer_name):
    """Zwraca wartość 'amount' dla pierwszego pasującego ID, a jeśli brak – po nazwie."""
    for pid in prefer_id:
        for x in nutrients:
            n = x.get("nutrient", {})
            if n.get("id") == pid:
                return x.get("amount")
    for pname in prefer_name:
        for x in nutrients:
            n = x.get("nutrient", {})
            if n.get("name") == pname:
                return x.get("amount")
    return None

def choose_portion(portions):
    """
    Zwraca wybraną porcję z FoundationFoods.foodPortions:
      1) value == 1 i modifier == 'medium' (jeśli jest),
      2) w przeciwnym razie dowolna porcja z value == 1,
      3) fallback: 1 g (unit: gram, symbol: g, usdaUnitId = None).
    """
    for p in portions:
        if p.get("value") == 1 and p.get("modifier", "").lower() == "medium":
            return p
    for p in portions:
        if p.get("value") == 1:
            return p
    return {
        "measureUnit": {"id": None, "name": "gram", "abbreviation": "g"},
        "gramWeight": 1.0,
    }

# 3) Budowa listy rekordów w docelowym formacie JSON
rows = []
for food in foods:
    nutrients = food.get("foodNutrients", [])
    portion = choose_portion(food.get("foodPortions", []))
    mu = portion.get("measureUnit", {}) or {}

    rows.append({
        # finalne klucze:
        "usdaId": food.get("fdcId"),
        "name": food.get("description"),
        "unit": {
            "usdaUnitId": mu.get("id"),
            "name": mu.get("name"),
            "symbol": mu.get("abbreviation"),
        },
        "gramsPerUnit": portion.get("gramWeight"),

        # makra per 100 g
        "kcal100g":      pick_value(nutrients, **TARGETS["kcal"]),
        "protein100g":   pick_value(nutrients, **TARGETS["protein_g"]),
        "carbs100g":     pick_value(nutrients, **TARGETS["carb_g"]),
        "fat100g":       pick_value(nutrients, **TARGETS["fat_g"]),
    })

df = pd.DataFrame(rows)

# 4) Korekta ujemnych węglowodanów (artefakt "by difference" → obcinamy do zera)
neg_carbs_mask = df["carbs100g"] < 0
num_neg = int(neg_carbs_mask.sum())
if num_neg > 0:
    print(f"Skorygowano {num_neg} rekordów z ujemną ilością węglowodanów (ustawiono na 0).")
    df.loc[neg_carbs_mask, "carbs100g"] = 0
else:
    print("Brak rekordów wymagających korekty węglowodanów (wartości < 0).")

# 5) Ponowny sanity check na ujemne wartości makro (protein/fat/carbs)
invalid_rows = df[
    (df["protein100g"] < 0) |
    (df["fat100g"] < 0) |
    (df["carbs100g"] < 0)
]

if invalid_rows.empty:
    print("Brak rekordów z ujemnymi wartościami białka, tłuszczu lub węglowodanów.")
else:
    print(f"Wykryto {len(invalid_rows)} rekordów z błędnymi wartościami:")
    print(invalid_rows[["usdaId", "name", "protein100g", "fat100g", "carbs100g"]])

# 6) Podgląd (pierwsze 5) — czytelny wydruk
preview = df.copy()
preview["unitName"]   = preview["unit"].map(lambda u: (u or {}).get("name"))
preview["unitSymbol"] = preview["unit"].map(lambda u: (u or {}).get("symbol"))

cols = [
    "usdaId", "name", "unitName", "unitSymbol",
    "gramsPerUnit", "kcal100g", "protein100g", "carbs100g", "fat100g"
]
preview = preview[cols]

pd.set_option("display.max_colwidth", 60)

fmt = {
    "gramsPerUnit":  "{:.2f}".format,
    "kcal100g":      "{:.0f}".format,
    "protein100g":   "{:.2f}".format,
    "carbs100g":     "{:.2f}".format,
    "fat100g":       "{:.2f}".format,
}

print(preview.head(5).to_string(index=False, formatters=fmt))

# ładny json
print(json.dumps(df.head(5).to_dict(orient="records"), ensure_ascii=False, indent=2))


Skorygowano 8 rekordów z ujemną ilością węglowodanów (ustawiono na 0).
Brak rekordów z ujemnymi wartościami białka, tłuszczu lub węglowodanów.
 usdaId                                                     name unitName unitSymbol gramsPerUnit kcal100g protein100g carbs100g fat100g
 321358                                       Hummus, commercial     gram          g         1.00      229        7.35     14.90   17.10
 321360                                     Tomatoes, grape, raw      cup        cup       152.00       27        0.83      5.51    0.63
 321611 Beans, snap, green, canned, regular pack, drained solids      cup        cup       129.00       21        1.04      4.11    0.39
 323121                              Frankfurter, beef, unheated    piece      piece        48.60      314       11.70      2.89   28.00
 323294              Nuts, almonds, dry roasted, with salt added      cup        cup       135.00      620       20.40     16.20   57.80
[
  {
    "usdaId": 321358,
    "na

#### Sprawdzenie, gdzie USDA nie podało measure unit

In [29]:
mask = df["unit"].map(lambda u: (u or {}).get("usdaUnitId")).isna()
bad = df.loc[mask, ["usdaId","name","gramsPerUnit","kcal100g","protein100g","carbs100g","fat100g"]].copy()
bad["unitName"] = df.loc[mask, "unit"].map(lambda u: (u or {}).get("name"))
bad["unitSymbol"] = df.loc[mask, "unit"].map(lambda u: (u or {}).get("symbol"))

print(f"Liczba wierszy z pustym unit.usdaUnitId: {len(bad)}")
if not bad.empty:
    print(bad.head(20).to_string(index=False))



Liczba wierszy z pustym unit.usdaUnitId: 263
 usdaId                                               name  gramsPerUnit  kcal100g  protein100g  carbs100g  fat100g unitName unitSymbol
 321358                                 Hummus, commercial           1.0     229.0         7.35      14.90    17.10     gram          g
 327046                              Kiwifruit, green, raw           1.0      58.0         1.06      14.00     0.44     gram          g
 328637                                    Cheese, cheddar           1.0     408.0        23.30       2.44    34.00     gram          g
 334194 Fish, tuna, light, canned in water, drained solids           1.0      90.0        19.00       0.08     0.94     gram          g
 746764                        Carrots, frozen, unprepared           1.0      37.0         0.81       7.92     0.47     gram          g
 746777                       Sauce, salsa, ready-to-serve           1.0      29.0         1.44       6.74     0.19     gram          g
 74

Jak widać, jest tego dużo
#### Uzupełnienie brakujących unit.usdaUnitId (fallback = gram / 1 g)
#### unitID w USDA mieszą się miedzy 1000 a 2000, więc dodaję w mmiejsce 1g placeholder "2048"

In [30]:
mask = df["unit"].map(lambda u: (u or {}).get("usdaUnitId")).isna()
count = int(mask.sum())

df.loc[mask, "unit"] = df.loc[mask, "unit"].apply(
    lambda u: {"usdaUnitId": 2048, "name": "gram", "symbol": "g"}
)
df.loc[mask, "gramsPerUnit"] = 1.0

print(f"Uzupełniono {count} wierszy brakujących jednostek (ustawiono gram / 1 g).")



Uzupełniono 263 wierszy brakujących jednostek (ustawiono gram / 1 g).


#### Sprawdzenie, gdzie są jeszcze wybrakowane wartości

In [31]:
null_rows = df[df.isna().any(axis=1)].copy()

print(f"Liczba wierszy z brakami: {len(null_rows)}")
null_rows.head(20)


Liczba wierszy z brakami: 41


,usdaId,name,unit,gramsPerUnit,kcal100g,protein100g,carbs100g,fat100g
61,746775,"Salt, table, iodized","{'usdaUnitId': 1002, 'name': 'teaspoon', 'symbol': 'tsp'}",6.1,NaN,NaN,NaN,NaN
74,747430,"Beans, Dry, Medium Red (0% moisture)","{'usdaUnitId': 2048, 'name': 'gram', 'symbol': 'g'}",1.0,NaN,25.5,NaN,1.04
75,747431,"Beans, Dry, Red (0% moisture)","{'usdaUnitId': 2048, 'name': 'gram', 'symbol': 'g'}",1.0,NaN,21.3,NaN,1.16
76,747432,"Beans, Dry, Flor de Mayo (0% moisture)","{'usdaUnitId': 2048, 'name': 'gram', 'symbol': 'g'}",1.0,NaN,23.3,NaN,0.86
77,747433,"Beans, Dry, Brown (0% moisture)","{'usdaUnitId': 2048, 'name': 'gram', 'symbol': 'g'}",1.0,NaN,25.6,NaN,1.12
78,747434,"Beans, Dry, Tan (0% moisture)","{'usdaUnitId': 2048, 'name': 'gram', 'symbol': 'g'}",1.0,NaN,26.8,NaN,1.14
79,747435,"Beans, Dry, Light Tan (0% moisture)","{'usdaUnitId': 2048, 'name': 'gram', 'symbol': 'g'}",1.0,NaN,24.6,NaN,1.28
80,747436,"Beans, Dry, Carioca (0% moisture)","{'usdaUnitId': 2048, 'name': 'gram', 'symbol': 'g'}",1.0,NaN,25.2,NaN,1.44
81,747437,"Beans, Dry, Cranberry (0% moisture)","{'usdaUnitId': 2048, 'name': 'gram', 'symbol': 'g'}",1.0,NaN,24.4,NaN,1.23
82,747438,"Beans, Dry, Light Red Kidney (0% moisture)","{'usdaUnitId': 2048, 'name': 'gram', 'symbol': 'g'}",1.0,NaN,25.0,NaN,1.03


Mamy obecnie 41 produktów z brakującymi danymi.
Braki dotyczą wartości makroskładników (białka, tłuszczu lub węglowodanów), więc takie rekordy są bezużyteczne dla naszego celu.
Na tym etapie zostaną pominięte i nie trafią do finalnego zbioru danych w formacie JSON.


In [34]:
import json
import pandas as pd
import os

print("Bieżący katalog roboczy:", os.getcwd())

df_ready = df.dropna().copy()
df_ready.to_json("ready_data.json", orient="records", indent=2, force_ascii=False)

print(f"Zapisano {len(df_ready)} rekordów do pliku ready_data.json\n")

# Podgląd pierwszych 3 rekordów w formacie JSON
preview = df_ready.head(3).to_dict(orient="records")
print("Podgląd zapisanych danych (pierwsze 5):\n")
print(json.dumps(preview, ensure_ascii=False, indent=2))




Bieżący katalog roboczy: C:\Users\piotr\dieti_data\dYeti_data\jowisz
Zapisano 299 rekordów do pliku ready_data.json

Podgląd zapisanych danych (pierwsze 5):

[
  {
    "usdaId": 321358,
    "name": "Hummus, commercial",
    "unit": {
      "usdaUnitId": 2048,
      "name": "gram",
      "symbol": "g"
    },
    "gramsPerUnit": 1.0,
    "kcal100g": 229.0,
    "protein100g": 7.35,
    "carbs100g": 14.9,
    "fat100g": 17.1
  },
  {
    "usdaId": 321360,
    "name": "Tomatoes, grape, raw",
    "unit": {
      "usdaUnitId": 1000,
      "name": "cup",
      "symbol": "cup"
    },
    "gramsPerUnit": 152.0,
    "kcal100g": 27.0,
    "protein100g": 0.83,
    "carbs100g": 5.51,
    "fat100g": 0.63
  },
  {
    "usdaId": 321611,
    "name": "Beans, snap, green, canned, regular pack, drained solids",
    "unit": {
      "usdaUnitId": 1000,
      "name": "cup",
      "symbol": "cup"
    },
    "gramsPerUnit": 129.0,
    "kcal100g": 21.0,
    "protein100g": 1.04,
    "carbs100g": 4.11,
    "fat100

### Gotowe rekordy do importu – szybki podgląd

| usdaId | name                         | unit (usdaUnitId · name · symbol) | gramsPerUnit | kcal/100g | P g/100g | C g/100g | F g/100g |
|-------:|------------------------------|------------------------------------|-------------:|----------:|---------:|---------:|---------:|
| 323121 | Frankfurter, beef, unheated  | 1043 · piece · piece              | **48.60**    | 314       | 11.70    | 2.89     | 28.00    |
| 321360 | Tomatoes, grape, raw         | 1000 · cup · cup                  | **152.00**   | 27        | 0.83     | 5.51     | 0.63     |
| 328637 | Cheese, cheddar              | 2048 · gram · g                   | **1.00**   | 408       | 23.30    | 2.44     | 34.00    |

**Co widać?**
- Każdy rekord ma komplet pól zgodnych z docelowym JSON
- `unit` przechowuje metadane USDA (`usdaUnitId`) oraz nazwę i symbol; przelicznik porcji w gramach jest w `gramsPerUnit`.
- Wartości energetyczne i makro są podane **na 100 g** jadalnej części (zgodnie z USDA).

Plik **`ready_data.json`** zawiera analogiczne obiekty dla wszystkich produktów bez braków.



### Przykładowe rekordy z `ready_data.json`

```json
  {
    "usdaId":321358,
    "name":"Hummus, commercial",
    "unit":{
      "usdaUnitId":2137,
      "name":"gram",
      "symbol":"g"
    },
    "gramsPerUnit":1.0,
    "kcal100g":229.0,
    "protein100g":7.35,
    "carbs100g":14.9,
    "fat100g":17.1
  },

  {
    "usdaId":321611,
    "name":"Beans, snap, green, canned, regular pack, drained solids",
    "unit":{
      "usdaUnitId":1000,
      "name":"cup",
      "symbol":"cup"
    },
    "gramsPerUnit":129.0,
    "kcal100g":21.0,
    "protein100g":1.04,
    "carbs100g":4.11,
    "fat100g":0.39
  },

  {
    "usdaId":323121,
    "name":"Frankfurter, beef, unheated",
    "unit":{
      "usdaUnitId":1043,
      "name":"piece",
      "symbol":"piece"
    },
    "gramsPerUnit":48.6,
    "kcal100g":314.0,
    "protein100g":11.7,
    "carbs100g":2.89,
    "fat100g":28.0
  },

```